# Statistics

Permutation tests of every tested metric (one table per metric), and the numbers quoted in the text.

All metrics are computed per bird and averaged over birds (binary metrics as % of birds). The tests compare each condition to labels shuffled within each chunk; "corrected p" is Bonferroni-corrected per metric over all 29 conditions. Cited ranges are min–max over conditions, with annotated labels for BVS and NS, predicted labels for RT and BWPS-clean, and without raw BWPS. Shuffled values in the text come from the single fixed shuffle shown in the figures.

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd

from constants import RHYTHM_TYPES
from pipeline.collect_permutation_tests import load_permtest

In [2]:
tests = load_permtest()
metrics = pd.read_csv('../data/metrics.csv')

# z is undefined (inf) when every shuffle gives the same value, e.g. no isolated labels
tests['z'] = tests['z'].replace([np.inf, -np.inf], np.nan)

In [3]:
# Datasets and labels used for the numbers in the text (raw BWPS is left out)
MAIN_LABELS = {'bvs': 'annot', 'ns': 'annot', 'rt': 'pred', 'bwps-clean': 'pred'}

# The rhythm types shown in Figure 2
FIG2_RHYTHM_TYPES = ['syll_ioi', 'syll_dur', 'gap_dur', 'elem_ioi', 'elem_dur']

N_DRAWS = tests['n_draws'].iloc[0]
N_TESTS = tests.groupby('metric').size().iloc[0]
print(f"Smallest p: 1/{N_DRAWS + 1} = {1 / (N_DRAWS + 1):.4f}")
print(f"Smallest corrected p: {N_TESTS}/{N_DRAWS + 1} = {N_TESTS / (N_DRAWS + 1):.4f}")

Smallest p: 1/2001 = 0.0005
Smallest corrected p: 29/2001 = 0.0145


## Permutation tests

Per condition: the observed value, the mean of the shuffles, z = (observed − shuffled mean) / SD of the shuffles, p and corrected p.

- `inconsistency`: median within-label SD (ms), expected lower than shuffled
- `isolated_label`: % of birds with ≥1 isolated label (Cohen's d to its nearest label > 3), expected higher
- `cluster_inconsistency`: median over clusters of the mean SD of the first and second durations (ms), expected lower
- `isolated_cluster`: % of birds with ≥1 isolated cluster (Mahalanobis distance > 3), expected higher
- `least_isochronous`, `median_isochrony`, `most_isochronous`: highest, median and lowest |r − 0.5| of the centroids of a bird's clusters with ≥5% of its dyads, expected higher

In [4]:
def test_table(metric):
    """The tests of one metric, sorted by label type, dataset and rhythm type; non-significant rows in red."""
    table = tests.query('metric == @metric').reset_index()
    table['rhythm_type'] = table['rhythm_type'].map(lambda r: RHYTHM_TYPES[r]['label'])
    table = table.set_index(['label_type', 'dataset_id', 'rhythm_type']).sort_index(level='label_type', sort_remaining=False)
    table = table[['value', 'null_mean', 'z', 'p', 'p_bonf', 'n_birds']].round(4)

    def highlight(row):
        return ['background-color: #f8d7da' if row['p_bonf'] >= 0.05 else ''] * len(row)
    return table.style.apply(highlight, axis=1).format(precision=4)

### `inconsistency`

In [5]:
test_table('inconsistency')

### `isolated_label`

In [6]:
test_table('isolated_label')

### `cluster_inconsistency`

In [7]:
test_table('cluster_inconsistency')

### `isolated_cluster`

In [8]:
test_table('isolated_cluster')

### `least_isochronous`

In [9]:
test_table('least_isochronous')

### `median_isochrony`

In [10]:
test_table('median_isochrony')

### `most_isochronous`

In [11]:
test_table('most_isochronous')

### Summary

Per metric: significant conditions, and the range of |z| (undefined z left out), as quoted in the text.

In [12]:
summary = tests.assign(significant=tests['p_bonf'] < 0.05, abs_z=tests['z'].abs())
summary.groupby('metric').agg(
    significant=('significant', 'sum'),
    conditions=('significant', 'size'),
    abs_z_min=('abs_z', 'min'),
    abs_z_max=('abs_z', 'max'),
    z_undefined=('abs_z', lambda z: z.isna().sum()),
    max_corrected_p=('p_bonf', 'max'),
).round(4)

,significant,conditions,abs_z_min,abs_z_max,z_undefined,max_corrected_p
metric,,,,,,
cluster_inconsistency,29,29,16.9324,141.7759,0,0.0145
inconsistency,29,29,31.7919,217.4219,0,0.0145
isolated_cluster,29,29,38.0436,829.4377,1,0.0145
isolated_label,29,29,33.6318,962.1868,10,0.0145
least_isochronous,29,29,5.0980,70.3167,0,0.0145
median_isochrony,29,29,6.0243,50.3175,0,0.0145
most_isochronous,23,29,0.0220,34.3973,0,1.0000


In [13]:
# The same per label type
summary.groupby(['metric', 'label_type']).agg(
    significant=('significant', 'sum'),
    conditions=('significant', 'size'),
    abs_z_min=('abs_z', 'min'),
    abs_z_max=('abs_z', 'max'),
).round(1)

significant  conditions  abs_z_min  \
metric                label_type                                       
cluster_inconsistency annot                13          13       16.9   
                      pred                 16          16       22.3   
inconsistency         annot                13          13       31.8   
                      pred                 16          16       45.1   
isolated_cluster      annot                13          13       38.0   
                      pred                 16          16       46.6   
isolated_label        annot                13          13       33.6   
                      pred                 16          16       60.1   
least_isochronous     annot                13          13        5.1   
                      pred                 16          16        9.0   
median_isochrony      annot                13          13        6.0   
                      pred                 16          16       14.3   
most_isochronous      annot                 8          13        0.0   
                      pred                 15          16        3.0   

                                  abs_z_max  
metric                label_type             
cluster_inconsistency annot           137.3  
                      pred            141.8  
inconsistency         annot           187.0  
                      pred            217.4  
isolated_cluster      annot           670.8  
                      pred            829.4  
isolated_label        annot           728.5  
                      pred            962.2  
least_isochronous     annot            25.0  
                      pred             70.3  
median_isochrony      annot            24.1  
                      pred             50.3  
most_isochronous      annot            10.7  
                      pred             34.4

In [14]:
# Non-significant tests
tests.query('p_bonf >= 0.05')[['value', 'null_mean', 'z', 'p', 'p_bonf']].round(3)

value  null_mean      z  \
dataset_id rhythm_type label_type metric                                      
bvs        elem_bound  annot      most_isochronous  0.036      0.044 -0.708   
           elem_dur    annot      most_isochronous  0.033      0.033  0.022   
           elem_ioi    annot      most_isochronous  0.029      0.041 -1.526   
ns         syll_bound  annot      most_isochronous  0.079      0.018  4.494   
           syll_ioi    annot      most_isochronous  0.023      0.014  2.387   
                       pred       most_isochronous  0.025      0.014  3.022   

                                                        p  p_bonf  
dataset_id rhythm_type label_type metric                           
bvs        elem_bound  annot      most_isochronous  0.759   1.000  
           elem_dur    annot      most_isochronous  0.471   1.000  
           elem_ioi    annot      most_isochronous  0.949   1.000  
ns         syll_bound  annot      most_isochronous  0.002   0.058  
           syll_ioi    annot      most_isochronous  0.018   0.522  
                       pred       most_isochronous  0.004   0.130

## Numbers for the text

The values per condition are also in the supplementary tables (`figS4-S5-metrics.ipynb`), rounded.

In [15]:
# One row per cited condition: the main label type of each dataset, no raw BWPS
is_cited = metrics['dataset_id'].map(MAIN_LABELS) == metrics['label_type']
cited = metrics[is_cited].set_index(['dataset_id', 'rhythm_type'])

# The single fixed shuffle shown in the figures
shuffled = metrics.query("label_type == 'shuffle' and dataset_id in @MAIN_LABELS").set_index(['dataset_id', 'rhythm_type'])

assert len(cited) == len(shuffled) == 15

### Consistency

In [16]:
print("Variability: {:.1f}–{:.1f} ms".format(cited['inconsistency'].min(), cited['inconsistency'].max()))
shuffled_main = shuffled.query('rhythm_type in @FIG2_RHYTHM_TYPES')
print("Variability, shuffled labels (five main rhythm types): {:.1f}–{:.1f} ms".format(
    shuffled_main['inconsistency'].min(), shuffled_main['inconsistency'].max()))
print("Mean duration of syllable IOIs: {:.0f} ms".format(cited.xs('syll_ioi', level='rhythm_type')['dur_mean'].mean()))
print("Mean duration of gaps: {:.0f} ms".format(cited.xs('gap_dur', level='rhythm_type')['dur_mean'].mean()))

Variability: 2.9–5.6 ms
Variability, shuffled labels (five main rhythm types): 11.2–59.0 ms
Mean duration of syllable IOIs: 144 ms
Mean duration of gaps: 39 ms


In [17]:
cited['inconsistency'].unstack('dataset_id').round(1)

dataset_id,bvs,bwps-clean,ns,rt
rhythm_type,,,,
elem_bound,3.5,NaN,NaN,NaN
elem_dur,3.4,NaN,NaN,NaN
elem_ioi,3.0,NaN,NaN,NaN
gap_dur,4.2,NaN,3.0,NaN
gap_ioi,5.5,NaN,3.5,NaN
syll_bound,4.0,NaN,2.9,NaN
syll_dur,4.3,NaN,2.9,NaN
syll_ioi,3.6,5.6,3.3,4.7


### Differentiation

In [18]:
isolated = cited['isolated_label'].unstack('dataset_id')
print("Birds with isolated syllable IOIs: {:.0f}–{:.0f}%".format(isolated.loc['syll_ioi'].min(), isolated.loc['syll_ioi'].max()))
print("Birds with isolated gap durations: {:.0f}–{:.0f}%".format(isolated.loc['gap_dur'].min(), isolated.loc['gap_dur'].max()))
isolated.round(0)

Birds with isolated syllable IOIs: 70–100%
Birds with isolated gap durations: 48–53%


dataset_id,bvs,bwps-clean,ns,rt
rhythm_type,,,,
elem_bound,62.0,NaN,NaN,NaN
elem_dur,68.0,NaN,NaN,NaN
elem_ioi,86.0,NaN,NaN,NaN
gap_dur,48.0,NaN,53.0,NaN
gap_ioi,85.0,NaN,93.0,NaN
syll_bound,92.0,NaN,93.0,NaN
syll_dur,92.0,NaN,100.0,NaN
syll_ioi,94.0,85.0,100.0,70.0


### Clusters

In [19]:
print("Cluster variability: {:.1f}–{:.1f} ms".format(cited['cluster_inconsistency'].min(), cited['cluster_inconsistency'].max()))
print("Birds with isolated clusters: {:.0f}–{:.0f}%".format(cited['isolated_cluster'].min(), cited['isolated_cluster'].max()))

Cluster variability: 2.9–6.9 ms
Birds with isolated clusters: 62–100%


### Coexistence

Over the rhythm types of Figure 2. A non-isochrony d means ratios r = 0.5 ± d, where the longer duration is (0.5 + d) / (0.5 − d) times the shorter one.

In [20]:
fig2 = cited.query('rhythm_type in @FIG2_RHYTHM_TYPES')
most, least = fig2['most_isochronous'], fig2['least_isochronous']
print("Most isochronous pattern: {:.2f}–{:.2f}".format(most.min(), most.max()))
print("Least isochronous pattern: {:.2f}–{:.2f}".format(least.min(), least.max()))

for d in [least.min(), least.max()]:
    print(f"  d = {d:.2f}: r = {0.5 - d:.2f} or {0.5 + d:.2f}, longer duration {(0.5 + d) / (0.5 - d):.1f}x the shorter one")

Most isochronous pattern: 0.01–0.05
Least isochronous pattern: 0.17–0.31
  d = 0.17: r = 0.33 or 0.67, longer duration 2.0x the shorter one
  d = 0.31: r = 0.19 or 0.81, longer duration 4.2x the shorter one


In [21]:
fig2[['most_isochronous', 'least_isochronous']].round(3)

most_isochronous  least_isochronous
dataset_id rhythm_type                                     
bvs        elem_dur                0.033              0.264
           elem_ioi                0.029              0.308
           gap_dur                 0.052              0.167
           syll_dur                0.048              0.220
           syll_ioi                0.042              0.195
bwps-clean syll_ioi                0.028              0.209
ns         gap_dur                 0.034              0.216
           syll_dur                0.040              0.237
           syll_ioi                0.023              0.203
rt         syll_ioi                0.014              0.216

### Predominance of non-isochrony

In [22]:
print("Isochronous dyads at rho = 0.05: {:.0f}–{:.0f}%".format(cited['perc_rel_isochronous'].min(), cited['perc_rel_isochronous'].max()))
print("Isochronous dyads at tau = 20 ms: {:.0f}–{:.0f}%".format(cited['perc_abs_isochronous'].min(), cited['perc_abs_isochronous'].max()))
majority = cited.query('perc_rel_isochronous >= 50 or perc_abs_isochronous >= 50')
print("Conditions where most dyads are isochronous:", list(majority.index))
cited[['perc_rel_isochronous', 'perc_abs_isochronous', 'rho50', 'tau50']].round(2)

Isochronous dyads at rho = 0.05: 6–32%
Isochronous dyads at tau = 20 ms: 11–70%
Conditions where most dyads are isochronous: [('bvs', 'gap_dur'), ('ns', 'gap_dur')]


perc_rel_isochronous  perc_abs_isochronous  rho50  \
dataset_id rhythm_type                                                      
bvs        elem_bound                  16.97                 43.99   0.14   
           elem_dur                    18.41                 41.27   0.14   
           elem_ioi                    18.40                 29.27   0.18   
           gap_dur                     32.00                 70.05   0.08   
           gap_ioi                     31.20                 24.37   0.09   
           syll_bound                   7.31                 15.03   0.24   
           syll_dur                    26.74                 29.62   0.12   
           syll_ioi                    28.60                 25.30   0.10   
bwps-clean syll_ioi                    27.65                 20.79   0.11   
ns         gap_dur                     29.93                 61.61   0.09   
           gap_ioi                     31.36                 18.36   0.07   
           syll_bound                   5.81                 10.94   0.28   
           syll_dur                    25.05                 15.71   0.11   
           syll_ioi                    30.04                 16.89   0.08   
rt         syll_ioi                    30.57                 26.15   0.08   

                        tau50  
dataset_id rhythm_type         
bvs        elem_bound   22.96  
           elem_dur     23.91  
           elem_ioi     38.61  
           gap_dur      11.32  
           gap_ioi      50.07  
           syll_bound   58.35  
           syll_dur     45.30  
           syll_ioi     50.19  
bwps-clean syll_ioi     62.41  
ns         gap_dur      14.00  
           gap_ioi      48.00  
           syll_bound   92.90  
           syll_dur     57.00  
           syll_ioi     60.80  
rt         syll_ioi     45.89

### Repetitions

In [23]:
print("Dyads that repeat a label: {:.0f}–{:.0f}%".format(cited['repetitions'].min(), cited['repetitions'].max()))
cited['repetitions'].unstack('dataset_id').round(0)

Dyads that repeat a label: 0–14%


dataset_id,bvs,bwps-clean,ns,rt
rhythm_type,,,,
elem_bound,0.0,NaN,NaN,NaN
elem_dur,3.0,NaN,NaN,NaN
elem_ioi,1.0,NaN,NaN,NaN
gap_dur,2.0,NaN,0.0,NaN
gap_ioi,2.0,NaN,0.0,NaN
syll_bound,0.0,NaN,0.0,NaN
syll_dur,11.0,NaN,0.0,NaN
syll_ioi,2.0,10.0,0.0,14.0
